# Snapget y1 — fine-tune YOLO11 xác minh ảnh quest (Google Colab)

Notebook **mỏng**: mọi logic nằm trong `ml/snapget12/` + `ml/scripts/10–16` của repo. Kế hoạch: `Snapget/.claude/QUEST_AI_PLAN.md`; đặc tả dữ liệu VN: `ml/DATASET_SPEC.md`.

**Runtime → Change runtime type → T4 GPU.** Trình tự:

| Ô | Việc | Khi nào chạy |
|---|---|---|
| 0 | Mount Drive, clone repo, cài thư viện | **mỗi phiên** |
| 1 | COCO → YOLO (`10`) — lần đầu tải ~20–40', zip lên Drive; các phiên sau chỉ giải nén | mỗi phiên |
| 2 | Kaggle (API) + giải nén ảnh Google đã gán + tập VN | mỗi phiên |
| 3 | (tuỳ chọn) Pre-label ảnh chưa có box (`12`) → sửa tay trên Roboflow | khi có ảnh mới |
| 4 | Gộp + kiểm tra tập VN (`11`) | mỗi phiên (vài phút) |
| 5 | Train (`13`) — có thể qua 2–3 phiên, dùng ô **resume** | |
| 6 | Đánh giá + ablation (`14`) | sau train |
| 7 | Chọn lớp ra đề (`15`) | sau 6 |
| 8 | Export ONNX + upload HF (`16`) | khi y1 tốt hơn v0 trên ảnh thật |

> ⚠️ Colab ngắt ⇒ mất mọi thứ trên `/content`. Dữ liệu gốc + checkpoint luôn nằm trên **Drive** (`WORK`); dataset làm việc giải nén ra `/content/yolo` (đọc ảnh từ Drive chậm ~10×).

In [ ]:
# 0a. Mount Drive
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/snapget-ml'      # thư mục gốc trên Drive (đổi nếu muốn)
YOLO = '/content/yolo'                          # dataset làm việc (đĩa local Colab)
EXT = '/content/ext'                            # dữ liệu ngoài đã giải nén
import os
for d in [WORK, f'{WORK}/data', f'{WORK}/artifacts', YOLO, EXT]:
    os.makedirs(d, exist_ok=True)

In [ ]:
# 0b. Code + thư viện (đổi URL repo nếu cần)
%cd /content
!rm -rf datn && git clone --depth 1 https://github.com/HuyenVt001/datn.git datn
%cd /content/datn/ml
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. COCO 2017 → YOLO 12 lớp
Lần đầu: tải bằng FiftyOne rồi **zip lên Drive** (`coco_yolo.zip`, vài GB). Phiên sau: chỉ giải nén (~3–5').
Ảnh trong `coco/images` là symlink tới cache FiftyOne — `zip -r` mặc định **theo** symlink nên file zip chứa ảnh thật.

In [ ]:
import os
ZIP = f'{WORK}/data/coco_yolo.zip'
if os.path.exists(ZIP):
    !unzip -q -o {ZIP} -d {YOLO}
else:
    !python scripts/10_export_coco_yolo.py --out {YOLO}/coco --max-per-class 2000 --negatives 1500 \
        --test-max 3000 --test-negatives 500 --portrait-frac 0.2
    !cd {YOLO} && zip -q -r {ZIP} coco
!head -40 {YOLO}/coco/stats.json

## 2. Dữ liệu ngoài: Kaggle + ảnh Google đã gán + tập VN
- **Kaggle**: tạo API token (kaggle.com → Settings → *Create New Token*) → đặt `kaggle.json` vào `WORK/` trên Drive. **Không commit, không dán vào chat.** Điền danh sách dataset (dạng `owner/dataset-name`) — chọn dataset **có box**, xem `ml/DATASET_SPEC.md` mục G.
- **Ảnh Google đã gán nhãn** (export YOLO từ Roboflow, kèm `data.yaml`): zip → `WORK/data/google_labeled.zip`.
- **Tập VN** đúng cấu trúc `DATASET_SPEC.md` mục E: zip → `WORK/data/vn.zip`.
- **Ảnh cào + gán nhãn tự động**: chạy notebook `snapget_crawl_autolabel.ipynb` trước → nằm ở `WORK/data/vn_web/` (tự được dùng ở ô 4).

In [ ]:
KAGGLE_DATASETS = [
    # 'owner/dataset-name',   # <- điền vào, mỗi dataset 1 dòng
]
import os, shutil
if KAGGLE_DATASETS:
    os.makedirs('/root/.kaggle', exist_ok=True)
    shutil.copy(f'{WORK}/kaggle.json', '/root/.kaggle/kaggle.json'); os.chmod('/root/.kaggle/kaggle.json', 0o600)
    for i, ds in enumerate(KAGGLE_DATASETS):
        !kaggle datasets download -d {ds} -p {EXT}/kaggle{i} --unzip -q
for name in ['google_labeled', 'vn']:
    z = f'{WORK}/data/{name}.zip'
    if os.path.exists(z):
        !unzip -q -o {z} -d {EXT}/{name}
    else:
        print(f'(chưa có {z} — bỏ qua)')
# Ảnh cào + gán nhãn tự động (notebook snapget_crawl_autolabel.ipynb) nằm trên Drive -> chép ra đĩa local cho nhanh
if os.path.isdir(f'{WORK}/data/vn_web/train'):
    !rsync -a --exclude review {WORK}/data/vn_web/ {EXT}/vn_web/
!ls {EXT}

## 3. (Tuỳ chọn) Pre-label ảnh chưa có box → sửa tay
Đặt ảnh thô trên Drive (vd `WORK/raw/google/<lớp>/*.jpg`, `WORK/raw/self_test/*.jpg`). Script dùng **YOLO11x** gán box nháp → upload thư mục kết quả lên **Roboflow**, sửa 100% ảnh (đặc biệt các ảnh trong `needs_review.txt`), export YOLOv8 **không resize** (DATASET_SPEC mục D).

In [ ]:
RAW = f'{WORK}/raw/google'                 # thư mục ảnh thô cần pre-label
OUT = f'{WORK}/prelabel/google'
!python scripts/12_prelabel.py --images {RAW} --out {OUT} --model yolo11x.pt --conf 0.25 --imgsz 1280
!head -20 {OUT}/needs_review.txt

## 4. Gộp + kiểm tra tập VN → `data.yaml`
Lỗi ở tập VN ⇒ script **dừng** và in danh sách — sửa dữ liệu rồi chạy lại. Cảnh báo (độ đa dạng…) ghi ở `merge_report.json`.
Thêm/bớt nguồn bằng `--yolo DIR` (YOLO), `--coco-json JSON IMGDIR`, `--voc DIR`.

In [ ]:
import glob, os

def find_vn(base):
    # zip có thể chứa sẵn thư mục vn/ hoặc giải nén thẳng train/ test/
    for cand in (f'{base}/vn', base):
        if os.path.isdir(f'{cand}/train') or os.path.isdir(f'{cand}/test'):
            return cand
    return None

sources = ''
for d in sorted(glob.glob(f'{EXT}/kaggle*')) + [f'{EXT}/google_labeled']:
    if os.path.isdir(d):
        sources += f' --yolo {d}'     # dataset COCO-JSON/VOC: đổi thành --coco-json / --voc
VN_DIR = find_vn(f'{EXT}/vn')
vn = f'--vn {VN_DIR}' if VN_DIR else ''
if os.path.isdir(f'{EXT}/vn_web/train'):
    vn += f' --vn {EXT}/vn_web'     # ảnh cào tự động (chỉ train/)
!python scripts/11_merge_external.py --root {YOLO} --coco {YOLO}/coco {sources} {vn} --class-map yolo_class_map.yaml
!cp {YOLO}/merge_report.json {WORK}/artifacts/merge_report.json

## 5. Train y1 (YOLO11s) — ~8–12 phút/epoch trên T4
Checkpoint lưu thẳng lên Drive mỗi epoch. **Colab ngắt giữa chừng** → mở lại notebook, chạy ô 0 → 1 → 2 → 4, rồi chạy ô **5b (resume)** — KHÔNG chạy lại 5a.

In [ ]:
# 5a. Train mới
!python scripts/13_train_yolo.py --data {YOLO}/data.yaml --model yolo11s.pt \
    --project {WORK}/artifacts/yolo --name y1-s --epochs 40 --export-best {WORK}/artifacts/y1-s/best.pt

In [ ]:
# 5b. Resume sau khi Colab ngắt (cùng --project/--name với 5a)
!python scripts/13_train_yolo.py --data {YOLO}/data.yaml --project {WORK}/artifacts/yolo --name y1-s --resume \
    --export-best {WORK}/artifacts/y1-s/best.pt

In [ ]:
# 5c. (Ablation, tuỳ chọn) YOLO11n — nhẹ hơn ~3×, so latency/độ chính xác
!python scripts/13_train_yolo.py --data {YOLO}/data.yaml --model yolo11n.pt \
    --project {WORK}/artifacts/yolo --name y1-n --epochs 40 --export-best {WORK}/artifacts/y1-n/best.pt

## 6. Đánh giá + ablation (cùng tập, cùng quy trình chọn ngưỡng)
- `yolo11s-coco`: YOLO11s COCO gốc, **không fine-tune** (tự cắt head còn 12 lớp) — baseline.
- `v0`: model MobileNetV3 đang chạy production (`artifacts/v0/model.onnx`, nếu còn trên Drive).
- Ngưỡng chọn trên **real-calib** (ảnh thật), precision ≥ 0.80; chưa có tập VN thì tạm chọn trên val.

In [ ]:
import os
models = f'--model yolo11s-coco=yolo11s.pt --model y1-s={WORK}/artifacts/y1-s/best.pt'
if os.path.exists(f'{WORK}/artifacts/y1-n/best.pt'):
    models += f' --model y1-n={WORK}/artifacts/y1-n/best.pt'
if os.path.exists(f'{WORK}/artifacts/v0/model.onnx'):
    models += f' --classifier v0={WORK}/artifacts/v0/model.onnx'
meta = f'--vn-meta {VN_DIR}/metadata.csv' if VN_DIR and os.path.exists(f'{VN_DIR}/metadata.csv') else ''
!python scripts/14_eval_yolo.py --root {YOLO} --out {WORK}/artifacts/eval {models} {meta} --max-coco 2000

In [ ]:
# Xem nhanh: confusion matrix (detection) + confusion cấp ảnh + PR/F1 curve của y1-s trên ảnh thật
from IPython.display import Image, display
import os
for p in [f'{WORK}/artifacts/eval/y1-s/det_real_test/confusion_matrix_normalized.png',
          f'{WORK}/artifacts/eval/y1-s/image_confusion_real_test.png',
          f'{WORK}/artifacts/eval/y1-s/pr_curves_real_test.png',
          f'{WORK}/artifacts/eval/y1-s/det_real_test/BoxF1_curve.png']:
    if os.path.exists(p):
        print(p); display(Image(p, width=900))

## 7. Chọn lớp ra đề quest
Tiêu chí (sửa bằng `--criteria`): AP ảnh ≥ 0.85 · F1 ≥ 0.85 · AP50 box ≥ 0.75 · từ chối sai ≤ 15% · bị nhầm ≤ 5% · ≥ 20 ảnh test. In ra bảng cho báo cáo + đoạn `AI_QUEST_CLASSES` cho server. **Gửi Claude `class_selection.json` + `metrics_real_test.json` + ảnh confusion matrix.**

In [ ]:
!python scripts/15_select_classes.py --eval {WORK}/artifacts/eval/y1-s --split real_test

## 8. Export ONNX fp32 + upload HF Model repo
Chỉ khi y1 **tốt hơn v0 trên real_test**. Cần token HF quyền **write**. Sau đó restart Cloud Run (lệnh in ở cuối) — `/health` phải trả `modelVersion: "y1"`, `arch: "yolo"`.

In [ ]:
from huggingface_hub import login
login()                                        # dán token write
MODEL_REPO = 'Flowerng/snapget-ai-model'       # HF Model repo (miễn phí) — Cloud Run tải từ đây
!python scripts/16_export_yolo_onnx.py --weights {WORK}/artifacts/y1-s/best.pt \
    --thresholds {WORK}/artifacts/eval/y1-s/thresholds.json --version y1 --out {WORK}/artifacts/y1-s/export \
    --upload {MODEL_REPO}

## 9. Bảng tổng hợp cho báo cáo

In [ ]:
import json
rows = json.load(open(f'{WORK}/artifacts/eval/ablation.json'))
f = lambda v: '-' if v is None else (f'{v:.3f}' if isinstance(v, float) else str(v))
print(f"{'model':14s} {'split':10s} {'n':>5s} {'mAP ảnh':>8s} {'macroF1':>8s} {'AP50 box':>9s} {'AP50-95':>8s} {'p50 ms':>7s}")
for r in rows:
    print(f"{r['model']:14s} {r['split']:10s} {r['n']:5d} {f(r['imageMAP']):>8s} {f(r['macroF1']):>8s} "
          f"{f(r['detMAP50']):>9s} {f(r['detMAP50_95']):>8s} {f(r['latencyP50ms']):>7s}")